1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



🟩 2. Definición FINAL de grupos (ya no incluye categóricas tipo texto)

In [ ]:
# ============================================================
# GRUPO A — TOKENS
# ============================================================
token_vars_log = [
    "n_tokens_content",
    "average_token_length"
]

token_vars_scale = [
    "n_tokens_title",
    "n_unique_tokens",
    "n_non_stop_words",
    "n_non_stop_unique_tokens"
]


# ============================================================
# GRUPO B — MULTIMEDIA
# ============================================================
multimedia_count_vars_log = ["num_imgs", "num_videos"]
multimedia_binary_vars = ["has_imgs", "has_videos"]


# ============================================================
# GRUPO C — KEYWORDS
# ============================================================
kw_cols = [
    "kw_min_min","kw_max_min","kw_avg_min",
    "kw_min_max","kw_max_max","kw_avg_max",
    "kw_min_avg","kw_max_avg","kw_avg_avg"
]

keyword_vars_log = kw_cols + ["num_keywords"]
keyword_score_var = ["keyword_strength_score"]


# ============================================================
# GRUPO D — CANALES Y DÍAS
# ============================================================
channel_dummy_vars = [
    "data_channel_is_lifestyle","data_channel_is_entertainment",
    "data_channel_is_bus","data_channel_is_socmed",
    "data_channel_is_tech","data_channel_is_world"
]

weekday_dummy_vars = [
    "weekday_is_monday","weekday_is_tuesday","weekday_is_wednesday",
    "weekday_is_thursday","weekday_is_friday","weekday_is_saturday",
    "weekday_is_sunday","is_weekend"
]

group_d_vars = channel_dummy_vars + weekday_dummy_vars


# ============================================================
# GRUPO E — SENTIMIENTO
# ============================================================
sentiment_vars = [
    "global_subjectivity","global_sentiment_polarity",
    "global_rate_positive_words","global_rate_negative_words",
    "avg_positive_polarity","min_positive_polarity","max_positive_polarity",
    "avg_negative_polarity","min_negative_polarity","max_negative_polarity",
    "title_subjectivity","title_sentiment_polarity",
    "abs_title_subjectivity","abs_title_sentiment_polarity"
]

sentiment_vars_robust = [
    "global_rate_negative_words",
    "global_rate_positive_words",
]

sentiment_vars_minmax = list(set(sentiment_vars) - set(sentiment_vars_robust))


# ============================================================
# GRUPO F — SELF-REFERENCE + TIEMPO
# ============================================================
selfref_vars_log = [
    "self_reference_min_shares",
    "self_reference_max_shares",
    "self_reference_avg_sharess",
    "num_self_hrefs",
    "internal_links_log",
    "authority_score"
]

selfref_time_vars_scale = [
    "timedelta",
    "is_old_article"
]


# ============================================================
# GRUPO G — LDA TOPICS
# ============================================================
lda_vars = ["LDA_00","LDA_01","LDA_02","LDA_03","LDA_04"]


🟧 3. PIPELINES BASE (log → scaling, robust, minmax)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer


# Pipeline: imputación → log1p → minmax
log_then_minmax = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", MinMaxScaler()),
])


# Pipeline: imputación → minmax
only_minmax = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
])


# Pipeline robusto (para colas largas)
robust_scale = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])


# Pipeline para variables binarias
binary_passthrough = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])


🟨 4. ColumnTransformer FINAL (sin url, sin fugas, sin dummies mal tratados)

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("tokens_log", log_then_minmax, token_vars_log),
        ("tokens_scale", only_minmax, token_vars_scale),

        ("multimedia_log", log_then_minmax, multimedia_count_vars_log),
        ("multimedia_bin", binary_passthrough, multimedia_binary_vars),

        ("keywords_log", log_then_minmax, keyword_vars_log),
        ("keyword_score", only_minmax, keyword_score_var),

        ("group_d_dummies", binary_passthrough, group_d_vars),

        ("sentiment_minmax", only_minmax, sentiment_vars_minmax),
        ("sentiment_robust", robust_scale, sentiment_vars_robust),

        ("selfref_log", log_then_minmax, selfref_vars_log),
        ("selfref_time", only_minmax, selfref_time_vars_scale),

        ("lda_scale", only_minmax, lda_vars),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)


🟩 5. Pipeline final con el modelo (RandomForest como baseline de experimentos MLOps)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipeline_final = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )),
])
